In [ ]:
! pip install lxml
! pip install pydub
! pip install pympi-ling

  Using cached lxml-5.3.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (3.7 kB)
Using cached lxml-5.3.1-cp311-cp311-manylinux_2_28_x86_64.whl (5.0 MB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pydub-0.25.1-py2.py3-none-any.whl (32 kB)


# Fonction formatage dossiers corpus

In [3]:
import os
import shutil

eslo_raw_data = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO1_REPAS"
eslo_dataset = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS"
eslo_dataset_wav = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/wav"
eslo_dataset_trs = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/trs"

os.makedirs(eslo_dataset, exist_ok = True)
os.makedirs(eslo_dataset_trs, exist_ok = True)
os.makedirs(eslo_dataset_wav, exist_ok = True)

for root, dirs, files in os.walk(eslo_raw_data):
    for filename in files:
        if filename.lower().endswith(".wav") and '_22km.wav' not in filename:
            source_path = os.path.join(root, filename)
            destination_path = os.path.join(eslo_dataset_wav, filename)
            shutil.copy2(source_path, destination_path)
        if filename.endswith("_C.trs"):
            source_path = os.path.join(root, filename)
            filename_out = filename.rstrip('_C.trs')+'.trs'
            destination_path = os.path.join(eslo_dataset_trs, filename_out)
            shutil.copy2(source_path, destination_path)

# Fonction mp4 to wav

In [ ]:
from pydub import AudioSegment

# Fonction pour convertir mp4 en wav
def convert_mp4_to_wav(mp4_file, output_audio_dir):
    wav_file = os.path.join(output_audio_dir, os.path.basename(mp4_file).replace('.mp4', '.wav'))
    
    # Convertir le fichier audio en wav
    audio = AudioSegment.from_file(mp4_file, format="mp4")
    audio.export(wav_file, format="wav")
    
    return wav_file

# EAF CAENNAIS

In [16]:
import pympi.Elan as elan
import csv
import os
import glob
from pydub import AudioSegment

def extract_annotations(eaf_file):
    """Extrait les annotations d'un fichier EAF sous forme de liste."""
    eaf = elan.Eaf(eaf_file)
    annotations = []
    
    for tier in eaf.get_tier_names():
        if "EXTRA" in tier:
            tier = tier.rstrip('_EXTRA')
        
        for start, end, text in eaf.get_annotation_data_for_tier(tier):
            if (tier, start, end, text) not in annotations:
                annotations.append((tier, start, end, text))
    
    # Trier les annotations par timestamp_start puis timestamp_end
    annotations.sort(key=lambda x: (x[1], x[2]))
    
    return annotations

def split_audio(audio_file, output_dir, segment_duration=600000):
    """Divise un fichier audio en segments de 10 minutes maximum et enregistre les morceaux."""
    audio = AudioSegment.from_wav(audio_file)
    audio_segments = []
    
    for i, start in enumerate(range(0, len(audio), segment_duration)):
        end = min(start + segment_duration, len(audio))
        segment_audio = audio[start:end]
        segment_filename = os.path.join(output_dir, f"{os.path.basename(audio_file).replace('.wav', '')}_part{i+1}.wav")
        segment_audio.export(segment_filename, format="wav")
        audio_segments.append((segment_filename, start, end))
    
    return audio_segments

def create_dataset(annotations, audio_dir, eaf_file, output_file, cropped_audio_dir):
    """Crée un dataset TSV avec une ligne par segment audio."""
    segments = []
    
    # Identifier le fichier audio associé
    audio_file = os.path.join(audio_dir, os.path.basename(eaf_file).replace('.eaf', '.wav'))
    audio_segments = split_audio(audio_file, cropped_audio_dir)
    
    for segment_file, segment_start, segment_end in audio_segments:
        speakers = []    
        start_times = []
        end_times = []
        texts = []
        seen_annotations = set()
        
        for tier_name, start, end, text in annotations:
            if segment_start <= start < segment_end and (tier_name, start, end, text) not in seen_annotations:
                speakers.append(tier_name)
                start_times.append((start - segment_start) / 1000)  # Convertir en secondes
                end_times.append((end - segment_start) / 1000)
                texts.append(text)
                seen_annotations.add((tier_name, start, end, text))
        
        if speakers:
            segments.append([speakers, start_times, end_times, texts, segment_file])
    
    # Écrire dans le fichier TSV
    with open(output_file, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter='\t')
        writer.writerow(['speakers', 'timestamps_start', 'timestamps_end', 'texts', 'audio'])
        
        for segment in segments:
            writer.writerow(segment)
    
    print(f"Dataset sauvegardé dans {output_file}")

eaf_dir = "/mnt/PROJET_CAENNAIS/corpus/GC/2_diarisation/gold"
audio_dir = "/mnt/PROJET_CAENNAIS/corpus/GC/1_enregistrements/wav"
output_dir = "/home/ziane212/projects/finetuning_pyannote/CAENNAIS_datasets"
cropped_audio_dir = "/mnt/PROJET_CAENNAIS/corpus/GC/1_enregistrements/wav/cropped_audio"

os.makedirs(cropped_audio_dir, exist_ok=True)

eaf_files = glob.glob(os.path.join(eaf_dir, "*.eaf"))
for eaf_file in eaf_files:
    annotations = extract_annotations(eaf_file)
    output_file = os.path.join(output_dir, os.path.basename(eaf_file).replace('.eaf', '.tsv'))
    create_dataset(annotations, audio_dir, eaf_file, output_file, cropped_audio_dir)

Dataset sauvegardé dans /home/ziane212/projects/finetuning_pyannote/CAENNAIS_datasets/GC_E1.tsv


# Dataset à partir de fichiers EAF

In [ ]:
import pympi.Elan as elan
import csv
import os
import glob
import random
from pydub import AudioSegment

def extract_annotations(eaf_file):
    """Extrait les annotations d'un fichier EAF sous forme de liste."""
    eaf = elan.Eaf(eaf_file)
    annotations = []
    
    for tier in eaf.get_tier_names():
        if "extra" in tier.lower():
            # tier = tier.rstrip('_EXTRA')
            pass
        elif "norme" in tier.lower():
            pass
        else:
            for start, end, text in eaf.get_annotation_data_for_tier(tier):
                if (tier, start, end, text) not in annotations:
                    annotations.append((tier, start, end, text))
    
    # Trier les annotations par timestamp_start puis timestamp_end
    annotations.sort(key=lambda x: (x[1], x[2]))
    
    return annotations

def select_random_sample_annotations(annotations, sample_duration_ms):
    """Sélectionne aléatoirement des annotations pour atteindre une durée cible."""
    sampled_annotations = []
    total_duration = 0
    available_annotations = annotations.copy()
    
    while available_annotations and total_duration < sample_duration_ms:
        annotation = random.choice(available_annotations)
        duration = annotation[2] - annotation[1]
        
        if total_duration + duration <= sample_duration_ms:
            sampled_annotations.append(annotation)
            total_duration += duration
        
        available_annotations.remove(annotation)
    
    remaining_annotations = [ann for ann in annotations if ann not in sampled_annotations]
    
    return sampled_annotations, remaining_annotations

def select_sample_annotations(annotations, sample_duration_ms):
    """Sélectionne les premières annotations de la liste jusqu'à atteindre une durée cible."""
    sampled_annotations = []
    total_duration = 0
    
    for annotation in annotations:
        duration = annotation[2] - annotation[1]
        
        if total_duration + duration <= sample_duration_ms:
            sampled_annotations.append(annotation)
            total_duration += duration
        else:
            break
    
    remaining_annotations = annotations[len(sampled_annotations):]
    
    return sampled_annotations, remaining_annotations

def split_audio_strict(audio_file, output_dir, segment_duration=300000):
    """Divise un fichier audio en segments de 10 minutes maximum et enregistre les morceaux."""
    audio = AudioSegment.from_wav(audio_file)
    audio_segments = []
    
    for i, start in enumerate(range(0, len(audio), segment_duration)):
        end = min(start + segment_duration, len(audio))
        segment_audio = audio[start:end]
        segment_filename = os.path.join(output_dir, f"{os.path.basename(audio_file).replace('.wav', '')}_part{i+1}.wav")
        segment_audio.export(segment_filename, format="wav")
        audio_segments.append((segment_filename, start, end))
    
    return audio_segments

def split_audio(audio_file, output_dir, annotations, segment_duration=300000, margin=10000):
    """Divise un fichier audio en segments en adaptant les coupes aux transitions de parole."""
    audio = AudioSegment.from_wav(audio_file)
    audio_segments = []
    
    start = 0
    i = 1

    while start < len(audio):
        end = start + segment_duration
        
        # Chercher la dernière annotation qui se termine avant end dans une marge raisonnable
        possible_cuts = [ann[2] for ann in annotations if start < ann[2] <= end]
        
        if possible_cuts:
            end = max(possible_cuts)  # Ajuster à la dernière annotation trouvée
        
        # Vérifier que la coupe ne dépasse pas la durée totale du fichier audio
        end = min(end, len(audio))
        
        segment_audio = audio[start:end]
        segment_filename = os.path.join(output_dir, f"{os.path.basename(audio_file).replace('.wav', '')}_part{i}.wav")
        segment_audio.export(segment_filename, format="wav")
        audio_segments.append((segment_filename, start, end))
        
        start = end  # Reprendre à la nouvelle coupe ajustée
        i += 1
    
    return audio_segments

def create_dataset(annotations, audio_dir, eaf_file, output_file, cropped_audio_dir, test_duration_ms):
    """Crée deux datasets TSV (train et test) en sélectionnant un échantillon et conservant le reste."""
    sampled_annotations, remaining_annotations = select_sample_annotations(annotations, test_duration_ms)
    
    for dataset_annotations, suffix in [(remaining_annotations, "train"), (sampled_annotations, "test")]:
        dataset_output_file = output_file.replace(".tsv", f"_{suffix}.tsv")
        segments = []
        
        # Identifier le fichier audio associé
        audio_file = os.path.join(audio_dir, os.path.basename(eaf_file).replace('.eaf', '.wav'))
        audio_segments = split_audio(audio_file, cropped_audio_dir, annotations)
        
        for segment_file, segment_start, segment_end in audio_segments:
            speakers = []    
            start_times = []
            end_times = []
            texts = []
            seen_annotations = set()
            
            for tier_name, start, end, text in dataset_annotations:
                if segment_start <= start < segment_end and (tier_name, start, end, text) not in seen_annotations:
                    speakers.append(tier_name)
                    start_times.append((start - segment_start) / 1000)  # Convertir en secondes
                    end_times.append((end - segment_start) / 1000)
                    texts.append(text)
                    seen_annotations.add((tier_name, start, end, text))
            
            if speakers:
                segments.append([speakers, start_times, end_times, texts, segment_file])
        
        # Écrire dans le fichier TSV
        with open(dataset_output_file, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f, delimiter='\t')
            writer.writerow(['speakers', 'timestamps_start', 'timestamps_end', 'texts', 'audio'])
            
            for segment in segments:
                writer.writerow(segment)
        
        print(f"Dataset sauvegardé dans {dataset_output_file}")

eaf_dir = "/mnt/PROJET_CAENNAIS/datasets_expe_frapeor/C"
audio_dir = "/mnt/PROJET_CAENNAIS/corpus/GC/1_enregistrements/wav"
output_dir = "/mnt/PROJET_CAENNAIS/datasets_expe_frapeor/datasets_v3"
cropped_audio_dir = "/mnt/PROJET_CAENNAIS/datasets_expe_frapeor/audio/cropped_audio_v3"

# test_duration_ms = 1800000
test_duration_ms = 0

os.makedirs(output_dir, exist_ok=True)
os.makedirs(cropped_audio_dir, exist_ok=True)

# Traitement des fichiers EAF
eaf_files = glob.glob(os.path.join(eaf_dir, "*.eaf"))
for eaf_file in eaf_files:
    annotations = extract_annotations(eaf_file)
    output_file = os.path.join(output_dir, os.path.basename(eaf_file).replace('.eaf', '.tsv'))
    create_dataset(annotations, audio_dir, eaf_file, output_file, cropped_audio_dir, test_duration_ms)

Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/datasets_expe_frapeor/datasets_v3/GC_E2_train.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/datasets_expe_frapeor/datasets_v3/GC_E2_test.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/datasets_expe_frapeor/datasets_v3/GC_E1_train.tsv


# Fabrication dataset à partir de fichiers EAF ou TRS

In [4]:
import pympi.Elan as elan
import csv
import os
import glob
import random
from pydub import AudioSegment
from lxml import etree

def extract_annotations(eaf_file):
    eaf = elan.Eaf(eaf_file)
    annotations = []

    for tier in eaf.get_tier_names():
        if "extra" in tier.lower() or "norme" in tier.lower():
            continue
        for start, end, text in eaf.get_annotation_data_for_tier(tier):
            if (tier, start, end, text) not in annotations:
                annotations.append((tier, start, end, text))

    annotations.sort(key=lambda x: (x[1], x[2]))
    return annotations

def extract_annotations_from_trs_final(trs_file):
    tree = etree.parse(trs_file)
    root = tree.getroot()
    annotations = []

    for turn in root.findall(".//Turn"):
        speaker = turn.attrib.get("speaker", "unknown")
        startTime = float(turn.attrib.get("startTime", 0)) * 1000
        endTime = float(turn.attrib.get("endTime", 0)) * 1000

        current_start = startTime
        buffer_text = ""

        for elem in turn:
            if elem.tag == "Sync":
                sync_time = float(elem.attrib["time"]) * 1000
                if buffer_text.strip():
                    annotations.append((speaker, current_start, sync_time, buffer_text.strip()))
                buffer_text = ""
                current_start = sync_time
                if elem.tail and elem.tail.strip():
                    buffer_text += elem.tail.strip() + " "

            elif elem.tag == "Event":
                if elem.text and elem.text.strip():
                    buffer_text += elem.text.strip() + " "
                if elem.tail and elem.tail.strip():
                    buffer_text += elem.tail.strip() + " "

            else:
                if elem.text and elem.text.strip():
                    buffer_text += elem.text.strip() + " "
                if elem.tail and elem.tail.strip():
                    buffer_text += elem.tail.strip() + " "

        if buffer_text.strip():
            annotations.append((speaker, current_start, endTime, buffer_text.strip()))

    annotations.sort(key=lambda x: (x[1], x[2]))
    return annotations

def extract_annotations_generic(file_path):
    if file_path.endswith(".eaf"):
        return extract_annotations(file_path)
    elif file_path.endswith(".trs"):
        return extract_annotations_from_trs_final(file_path)
    else:
        raise ValueError("Format de fichier non pris en charge. Utilisez .eaf ou .trs")

def select_sample_annotations(annotations, sample_duration_ms):
    sampled_annotations = []
    total_duration = 0
    for annotation in annotations:
        duration = annotation[2] - annotation[1]
        if total_duration + duration <= sample_duration_ms:
            sampled_annotations.append(annotation)
            total_duration += duration
        else:
            break
    remaining_annotations = annotations[len(sampled_annotations):]
    return sampled_annotations, remaining_annotations

def split_audio(audio_file, output_dir, annotations, segment_duration=300000, margin=10000):
    audio = AudioSegment.from_wav(audio_file)
    audio_segments = []
    start = 0
    i = 1
    while start < len(audio):
        end = start + segment_duration
        possible_cuts = [ann[2] for ann in annotations if start < ann[2] <= end]
        if possible_cuts:
            end = max(possible_cuts)
        end = min(end, len(audio))
        segment_audio = audio[start:end]
        segment_filename = os.path.join(output_dir, f"{os.path.basename(audio_file).replace('.wav', '')}_part{i}.wav")
        segment_audio.export(segment_filename, format="wav")
        audio_segments.append((segment_filename, start, end))
        start = end
        i += 1
    return audio_segments

def create_dataset(annotations, audio_dir, input_file, output_file, cropped_audio_dir, test_duration_ms):
    sampled_annotations, remaining_annotations = select_sample_annotations(annotations, test_duration_ms)
    for dataset_annotations, suffix in [(remaining_annotations, "train"), (sampled_annotations, "test")]:
        dataset_output_file = output_file.replace(".tsv", f"_{suffix}.tsv")
        segments = []
        audio_file = os.path.join(audio_dir, os.path.basename(input_file).replace('.eaf', '.wav').replace('.trs', '.wav'))
        audio_segments = split_audio(audio_file, cropped_audio_dir, annotations)
        for segment_file, segment_start, segment_end in audio_segments:
            speakers, start_times, end_times, texts = [], [], [], []
            seen_annotations = set()
            for tier_name, start, end, text in dataset_annotations:
                if segment_start <= start < segment_end and (tier_name, start, end, text) not in seen_annotations:
                    speakers.append(tier_name)
                    start_times.append((start - segment_start) / 1000)
                    end_times.append((end - segment_start) / 1000)
                    texts.append(text)
                    seen_annotations.add((tier_name, start, end, text))
            if speakers:
                segments.append([speakers, start_times, end_times, texts, segment_file])
        with open(dataset_output_file, mode='w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f, delimiter='\t')
            writer.writerow(['speakers', 'timestamps_start', 'timestamps_end', 'texts', 'audio'])
            for segment in segments:
                writer.writerow(segment)
        print(f"Dataset sauvegardé dans {dataset_output_file}")

# Exemple d'utilisation (adapter les chemins selon ton environnement)
input_dir = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/trs"
audio_dir = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/wav"
output_dir = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA"
os.makedirs(output_dir, exist_ok=True)

cropped_audio_dir = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/crop"
os.makedirs(cropped_audio_dir, exist_ok=True)

test_duration_ms = 0
input_files = glob.glob(os.path.join(input_dir, "*.eaf")) + glob.glob(os.path.join(input_dir, "*.trs"))
for input_file in input_files:
    annotations = extract_annotations_generic(input_file)
    output_file = os.path.join(output_dir, os.path.basename(input_file).replace('.eaf', '.tsv').replace('.trs', '.tsv'))
    create_dataset(annotations, audio_dir, input_file, output_file, cropped_audio_dir, test_duration_ms)


Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1256_train.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1256_test.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1264_train.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1264_test.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1270_train.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1270_test.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1263_train.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/ESLO2_REPAS_1263_test.tsv
Dataset sauvegardé dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/E

# concat datasets

In [5]:
import pandas as pd
import glob, os
def concat_tsv_files(tsv_files, output_file):
    """Concatène plusieurs fichiers TSV en un seul."""
    output_path = '/'.join(output_file.split('/')[:-1])
    os.makedirs(output_path, exist_ok=True)
    print(output_path)
    df_list = [pd.read_csv(file, sep='\t') for file in tsv_files]
    concatenated_df = pd.concat(df_list, ignore_index=True)
    concatenated_df.to_csv(output_file, sep='\t', index=False)
    print(f"Fichiers concaténés et sauvegardés dans {output_file}")

tsv_dir = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA"
output_file = "/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/all"
tsv_files = glob.glob(os.path.join(tsv_dir, "*.tsv"))
concat_tsv_files(tsv_files, output_file)
              

/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA
Fichiers concaténés et sauvegardés dans /mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/all


# split dataset

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd
import os

file_path = "/mnt/PROJET_CAENNAIS/commun/diarisation/data/eslo/ESLO_REPAS/dataset_DIA/all.tsv"

output_dir = "/mnt/PROJET_CAENNAIS/commun/diarisation/data/eslo/ESLO_REPAS/dataset_DIA_06.05.25/"
os.makedirs(output_dir, exist_ok=True)

# tsv_dir = "/mnt/PROJET_CAENNAIS/datasets_expe_frapeor/datasets/train"
# tsv_files = glob.glob(os.path.join(tsv_dir, "*.tsv"))
# Créer un DataFrame pandas pour les tableau TSV
# df_list = [pd.read_csv(file, sep='\t') for file in tsv_files]

df = pd.read_csv(file_path, sep='\t')

# Split data into train (80%) and test (20%)
train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

# Sauvegarder les tableaux en TSV
train_tsv_file = os.path.join(output_dir, "train_ESLO_all.tsv")
test_tsv_file = os.path.join(output_dir, "test_ESLO_all.tsv")
train_df.to_csv(train_tsv_file, sep='\t', index=False)
test_df.to_csv(test_tsv_file, sep='\t', index=False)

# split audio

In [ ]:
from pydub import AudioSegment
import os

def split_wav(file_path, segment_length=23 * 60 * 1000):
    """
    Coupe un fichier WAV en segments de durée spécifiée (par défaut 23 minutes).
    
    :param file_path: Chemin du fichier WAV d'entrée.
    :param segment_length: Durée d'un segment en millisecondes (23 min par défaut).
    """
    audio = AudioSegment.from_wav(file_path)
    file_name, ext = os.path.splitext(file_path)
    output_dir = f"{file_name}_splits"
    
    # Création du dossier de sortie
    os.makedirs(output_dir, exist_ok=True)

    # Découpage en segments
    for i, start_time in enumerate(range(0, len(audio), segment_length)):
        segment = audio[start_time:start_time + segment_length]
        segment.export(os.path.join(output_dir, f"{file_name}_part{i+1}.wav"), format="wav")
        print(f"Segment {i+1} exporté.")

    print(f"Tous les segments sont sauvegardés dans {output_dir}")

# Exemple d'utilisation
split_wav("/mnt/PROJET_CAENNAIS/corpus/GC/1_enregistrements/wav/GC_E2.wav")


Segment 1 exporté.
Segment 2 exporté.
Segment 3 exporté.
Tous les segments sont sauvegardés dans /mnt/PROJET_CAENNAIS/corpus/GC/1_enregistrements/wav/GC_E2_splits


# stats from dataset

In [9]:
import pandas as pd
from pydub.utils import mediainfo
from pathlib import Path

def format_duration(seconds):
    millis = int((seconds % 1) * 1000)
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h}:{m:02d}:{s:02d}.{millis:03d}"

path_in = "/mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/datasets_v3/test/GC_E2_train.tsv"
path_out = "/mnt/PROJET_CAENNAIS/commun/datasets_expe_frapeor_diarisation/datasets_v3/test/stats_duree_audio.tsv"
# Charger le fichier TSV
df = pd.read_csv(path_in, sep="\t")

# Extraire le nom de l'enregistrement (sans _partX) et le corpus
df["recording_id"] = df["audio"].apply(lambda x: Path(x).stem.split("_part")[0])
df["corpus"] = df["audio"].apply(lambda x: "ESLO1" if "ESLO1" in x else ("ESLO2" if "ESLO2" in x else "UNKNOWN"))

# Conserver le chemin du premier fichier rencontré par enregistrement
df_grouped = df.groupby(["recording_id", "corpus"]).agg(
    example_path=("audio", "first")
).reset_index()

# Récupérer la durée réelle de chaque fichier audio (_partX)
def get_audio_duration(filepath):
    try:
        info = mediainfo(filepath)
        return float(info['duration'])
    except Exception as e:
        print(f"Erreur avec {filepath} : {e}")
        return 0.0

# Récupérer tous les chemins d’audio à inclure pour chaque enregistrement
audio_parts = df[["recording_id", "audio"]].drop_duplicates()
durations = audio_parts.groupby("recording_id")["audio"].apply(
    lambda paths: sum(get_audio_duration(p) for p in paths)
).reset_index(name="total_duration_sec")

# Fusionner les durées avec les infos de base
final = df_grouped.merge(durations, on="recording_id")
final["formatted_duration"] = final["total_duration_sec"].apply(format_duration)

# Sauvegarde
final.to_csv(path_out, index=False)

# Totaux par corpus
totaux_sec = final.groupby("corpus")["total_duration_sec"].sum()
totaux_sec["TOTAL"] = final["total_duration_sec"].sum()
formatted_totaux = totaux_sec.apply(format_duration)

# Affichage
print("\nDurées totales par corpus (calculées sur les fichiers audio) :")
print(formatted_totaux)

print("\nAperçu :")
print(final[["recording_id", "corpus", "formatted_duration"]].head())



Durées totales par corpus (calculées sur les fichiers audio) :
corpus
UNKNOWN    0:23:22.764
TOTAL      0:23:22.764
Name: total_duration_sec, dtype: object

Aperçu :
  recording_id   corpus formatted_duration
0        GC_E2  UNKNOWN        0:23:22.764


# stats from audio dir

In [ ]:
import os
import wave
from pathlib import Path
from collections import defaultdict

def format_duration(seconds):
    millis = int((seconds % 1) * 1000)
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h}:{m:02d}:{s:02d}.{millis:03d}"

# Dossier contenant les fichiers .wav
wav_dir = Path("/mnt/PROJET_CAENNAIS/diarisation/data/eslo/ESLO_REPAS/wav")

# Initialisation
durations = []
corpus_totals = defaultdict(float)

# Parcourir les fichiers wav
for wav_file in wav_dir.glob("*.wav"):
    with wave.open(str(wav_file), "rb") as wf:
        frames = wf.getnframes()
        rate = wf.getframerate()
        duration_sec = frames / float(rate)
    
    filename = wav_file.name
    corpus = "ESLO1" if "ESLO1" in filename else ("ESLO2" if "ESLO2" in filename else "UNKNOWN")
    
    durations.append({
        "filename": filename,
        "corpus": corpus,
        "duration_sec": duration_sec,
        "formatted_duration": format_duration(duration_sec)
    })
    
    corpus_totals[corpus] += duration_sec

# Ajouter un total général
corpus_totals["TOTAL"] = sum(corpus_totals.values())

# Affichage
print("Durée par fichier :")
for item in durations:
    print(f"{item['filename']}: {item['formatted_duration']} ({item['corpus']})")

print("\nTotaux par corpus :")
for corpus, total in corpus_totals.items():
    print(f"{corpus}: {format_duration(total)}")
